In [2]:
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import joblib
import torch

# add path to project root
import sys
PROJECT_ROOT = Path.cwd()

if "notebooks" in PROJECT_ROOT.parts:
    PROJECT_ROOT = PROJECT_ROOT.parent
    
sys.path.append(str(PROJECT_ROOT))


from src.preprocess.gaming_text_preprocessor import GamingTextPreprocessor
from src.preprocess.gaming_text_labeler import GamingTextLabeler
from src.ensemble.ensemble import Ensemble
from src.model.bert_collection import BertToxicityModelCollection
from src.ensemble.score import ClassificationMetrics

# instantiate preprocessor and labeler
gaming_labeler = GamingTextLabeler()
metrics = ClassificationMetrics()



In [3]:
CONFIG = {
    'seed': 7524,
    'data_dir': PROJECT_ROOT / 'data' / 'processed_data',
    'model_dir': PROJECT_ROOT / 'models',
    'text_col': 'message',
    'label_col': 'label'
}

np.random.seed(CONFIG['seed'])
seed = CONFIG['seed']
tc, lc = CONFIG['text_col'], CONFIG['label_col']

print('CONFIG loaded. Text column:', tc)

CONFIG loaded. Text column: message


In [4]:
# load WOT train data
d = CONFIG['data_dir']

wot_train = pd.read_parquet(d / 'wot' / 'x_train.parquet')
wot_train["data_source"] = "wot"

# load DOTA training data
dota_train = pd.read_parquet(d / 'dota' / 'x_train.parquet')
dota_train["data_source"] = "dota"

# combine datasets
train = pd.concat([wot_train, dota_train], ignore_index=True)

# clean labels and create binary label column
train = gaming_labeler.convert_binary(
    train, 
    label_column=lc, 
    output_column='label_binary'
)

In [5]:
wot_val = pd.read_parquet(d / 'wot' / 'x_validation.parquet')
wot_val["data_source"] = "wot"
dota_val = pd.read_parquet(d / 'dota' / 'x_validation.parquet')
dota_val["data_source"] = "dota"

# combine holdout datasets
val = pd.concat([wot_val, dota_val], ignore_index=True)

# clean labels and create binary label column
val = gaming_labeler.convert_binary(
    val, 
    label_column=lc, 
    output_column='label_binary'
)

# Create Objects 

In [6]:
bert_binary_model = BertToxicityModelCollection(
    model_names=["dota_binary", "jigsaw_binary", "wot_binary"]
)

Loading dota_binary from jforward/bert-toxicity/dota_binary_model...


dota_binary_model/model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading jigsaw_binary from jforward/bert-toxicity/jigsaw_binary_model...


tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading wot_binary from jforward/bert-toxicity/wot_binary_model...


tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/815 [00:00<?, ?B/s]

wot_binary_model/model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

In [7]:
bert_binary_ensemble = Ensemble(
    model_collections=[bert_binary_model]
)

# Simple Majority Rules

In [8]:
pred_train = bert_binary_ensemble.predict_simple_majority(train[tc])
metrics.metrics(train['label_binary'], pred_train)

Predicting with SimpleMajority...


{'CV Macro F1': 0.8427718505432904,
 'CV Weighted F1': 0.8921975776052649,
 'Accuracy': 0.8993241104511516,
 'Coverage': 1.0,
 'Precision': 0.9226975570592958,
 'FPR': 0.016467541352097534,
 'FNR': 0.3704022538738457,
 'TPR': 0.6295977461261544,
 'TNR': 0.9835324586479025}

In [9]:
pred_train = bert_binary_ensemble.predict_simple_majority(val[tc])
metrics.metrics(val['label_binary'], pred_train)

Predicting with SimpleMajority...


{'CV Macro F1': 0.793306020748871,
 'CV Weighted F1': 0.8679214998907251,
 'Accuracy': 0.8775211069418386,
 'Coverage': 1.0,
 'Precision': 0.827922077922078,
 'FPR': 0.03175792075499963,
 'FNR': 0.4493927125506073,
 'TPR': 0.5506072874493927,
 'TNR': 0.9682420792450004}

# Confidence

In [ ]:
res = bert_binary_ensemble.fit_weighted_confidence_majority(
    X_val=train[tc], 
    y_val=train['label_binary'], 
    score_func=metrics.score,
    thresholds=thresholds
)

In [ ]:
combined_pred = bert_binary_ensemble.predict_weighted_confidence_majority(
    X=train[tc], 
    weights=res[0],
    thresholds=res[1]
)
metrics.metrics(train['label_binary'], combined_pred)

In [ ]:
combined_pred = bert_binary_ensemble.predict_weighted_confidence_majority(X=val[tc],
    weights=res[0],
    thresholds=res[1]
)
metrics.metrics(val['label_binary'], combined_pred)

In [ ]:
res